# Fit4Job — Model Evaluation
**YKA Rathnasiri | 27413 | BSc (Hons) Data Science**

Compares the three matching engines over the same 30 CVs and 1,711 job postings.

| Engine | Score |
|---|---|
| **Keyword** | skill-set overlap only (Jaccard over the taxonomy) |
| **SBERT** | semantic cosine only (SBERT → PCA → FAISS) |
| **Hybrid** | `0.7 × cosine + 0.3 × skill overlap` — the deployed system |

Each engine returns its top 10 per CV; every returned job is graded **A** good / **B**
partial / **C** not relevant; the grades give P@5, P@10, MRR and NDCG@10, averaged over
the 30 CVs.

> The Hybrid figures are carried over from `Model_Development.ipynb` and are **not**
> recomputed here, so the published results cannot shift.

## 1. Setup

In [1]:
import os, re, json, logging, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import faiss, joblib

warnings.filterwarnings('ignore')
logging.getLogger('pdfminer').setLevel(logging.ERROR)   # malformed-font noise

TOP_K   = 10
METRICS = ['P@5', 'P@10', 'MRR', 'NDCG@10']

# Hybrid results from Model_Development.ipynb §11 - reference only, never recomputed.
HYBRID = {'P@5': (0.9133, 0.1456), 'P@10': (0.8667, 0.1373),
          'MRR': (0.9611, 0.1496), 'NDCG@10': (0.9270, 0.0949)}

## 2. Data

Locates the job CSV, the skill taxonomy, the 30 resumes and the `pca_model.pkl` /
`faiss_index.bin` saved by `Model_Development.ipynb`. Reusing the saved index is what
keeps this evaluation on the same vector space the Hybrid was measured in.

In [2]:
ROOT  = next((p for p in Path.cwd().parents if (p / 'app').is_dir()), Path.cwd())
ROOTS = [ROOT, ROOT / 'data', ROOT / 'models_store', ROOT.parent,
         ROOT.parent / 'Fit4Job_contents' / 'model' / 'fit4job_models',
         ROOT.parent / 'Data Collection' / 'Resume', Path('/content')]
SKIP  = {'.git', '.venv', 'venv', 'node_modules', '__pycache__'}

def find(*names, depth=3):
    """First match for `names`, earlier names preferred across all roots."""
    for name in names:
        for root in ROOTS:
            if not root.exists():
                continue
            if (root / name).exists():
                return root / name
            hits = sorted(p for d in range(1, depth + 1)
                          for p in root.glob('/'.join(['*'] * d + [name]))
                          if not SKIP & set(p.parts))
            if hits:
                return hits[0]

JOBS_CSV = find('linkedin_it_jobs.csv', 'cleaned_jobs.csv')
TAX_JSON = find('it_skill_taxonomy.json', 'skill_taxonomy.json')

# several copies of the CV set exist; take the one with the most resumes
cv_dirs = {h.parent for n in ['AI & ML Engineer 1.pdf', 'Software Engineer 1.pdf']
           if (h := find(n))}
CV_DIR  = max(cv_dirs, key=lambda d: len(list(d.glob('*.pdf')) + list(d.glob('*.docx'))))

print(f'jobs     {JOBS_CSV}\ntaxonomy {TAX_JSON}\nresumes  {CV_DIR}')
assert JOBS_CSV and TAX_JSON and CV_DIR

jobs     D:\1. UNI\Final Reserch Project\Fit4Job_contents\model\fit4job_models\linkedin_it_jobs.csv
taxonomy D:\1. UNI\Final Reserch Project\Fit4Job_contents\model\fit4job_models\it_skill_taxonomy.json
resumes  D:\1. UNI\Final Reserch Project\Data Collection\Resume\Resume


In [3]:
with open(TAX_JSON, encoding='utf-8') as f:
    raw = json.load(f)

# accept both taxonomy layouts: {skill: [aliases]} and {skill: {'aliases': [...]}}
taxonomy = {k: ([a.lower() for a in v] if isinstance(v, list)
                else sorted({k.lower(), *(a.lower() for a in v.get('aliases', []))}))
            for k, v in raw.items()}
norm_map = {a: c for c, al in taxonomy.items() for a in al}

df = pd.read_csv(JOBS_CSV)
print(f'{len(df)} jobs | {len(taxonomy)} canonical skills | '
      f'{len(list(CV_DIR.glob("*.pdf")))} resumes')

1722 jobs | 750 canonical skills | 30 resumes


### Preprocessing

Identical to `Model_Development.ipynb` — drop non-English postings, strip HTML/noise,
extract canonical skills, build the text that gets embedded.

In [4]:
from langdetect import detect, DetectorFactory
from bs4 import BeautifulSoup

DetectorFactory.seed = 0

def is_english(t):
    try:
        return detect(t) == 'en' if isinstance(t, str) and len(t) > 20 else True
    except Exception:
        return True

def clean_text(t):
    if not isinstance(t, str) or not t.strip():
        return ''
    t = BeautifulSoup(t, 'html.parser').get_text(' ')
    t = re.sub(r'http\S+|www\S+', '', t)
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s\.\+\#\-/]', ' ', t)).strip().lower()

def extract_skills(text, tax=None):
    """Canonical skills found in `text`. Sorted, not set order - this list is
    concatenated into the embedded text, so an unstable order would move the
    vectors and every metric with them."""
    tax, tl, found = tax or taxonomy, str(text).lower(), set()
    for canonical, aliases in tax.items():
        for a in aliases:
            if re.search((r'\b' + re.escape(a) + r'\b') if len(a) <= 3 else re.escape(a), tl):
                found.add(canonical)
                break
    return sorted(found)

skills_col = 'required_skills' if 'required_skills' in df.columns else 'skills'
df[skills_col] = df[skills_col].fillna('')

n0 = len(df)
df = df[df['description'].apply(is_english)].reset_index(drop=True)

df['desc_clean'] = df['description'].apply(clean_text)
df['skills'] = df.apply(lambda r: sorted(set(
    [norm_map.get(s.lower().strip(), s.strip())
     for s in re.split(r'[,;]', str(r[skills_col])) if s.strip()]
    + extract_skills(r['desc_clean']))), axis=1)
df['embed_text'] = (df['title'].apply(clean_text) + ' ' + df['desc_clean'] + ' '
                    + df['skills'].apply(' '.join))

print(f'dropped {n0 - len(df)} non-english | corpus {len(df)} jobs | '
      f'avg {df["skills"].apply(len).mean():.0f} skills/job')

dropped 11 non-english | corpus 1711 jobs | avg 21 skills/job


## 3. Vectors and CVs

Loads the saved PCA model and FAISS index, then extracts and encodes the 30 resumes.

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

sbert = SentenceTransformer('all-MiniLM-L6-v2')

def encode(text, max_words=512):
    words = str(text).split()
    if not words:
        return np.zeros(sbert.get_sentence_embedding_dimension())
    if len(words) <= max_words:
        return sbert.encode(text, show_progress_bar=False)
    chunks = [' '.join(words[i:i + max_words]) for i in range(0, len(words), max_words)]
    return np.mean(sbert.encode(chunks, show_progress_bar=False), axis=0)

pca   = joblib.load(find('pca_model.pkl'))
index = faiss.read_index(str(find('faiss_index.bin')))
assert index.ntotal == len(df), f'index has {index.ntotal} vectors, corpus has {len(df)}'
print(f'index: {index.ntotal} vectors x {index.d} dims')

index: 1711 vectors x 128 dims


In [6]:
import pdfplumber, PyPDF2

def read_pdf(path):
    for how in ['plumber', 'pypdf2', 'miner']:
        try:
            if how == 'plumber':
                with pdfplumber.open(path) as p:
                    t = '\n'.join(pg.extract_text(x_tolerance=2, y_tolerance=2) or ''
                                  for pg in p.pages)
            elif how == 'pypdf2':
                with open(path, 'rb') as f:
                    t = '\n'.join(pg.extract_text() or ''
                                  for pg in PyPDF2.PdfReader(f).pages)
            else:
                from pdfminer.high_level import extract_text
                t = extract_text(path)
            if len(t.strip()) > 50:
                return t
        except Exception:
            pass
    return ''

cvs = []
for i, fn in enumerate(sorted(f for f in os.listdir(CV_DIR) if f.endswith('.pdf')), 1):
    cleaned = clean_text(read_pdf(str(CV_DIR / fn)))
    skills  = extract_skills(cleaned)
    cvs.append({'id': i, 'name': fn[:-4], 'text': cleaned, 'skills': skills,
                'embed_text': cleaned + ' ' + ' '.join(skills)})

cv_vecs = np.array([normalize(pca.transform(
    normalize(encode(c['embed_text']).reshape(1, -1), norm='l2')), norm='l2')[0]
    for c in cvs]).astype('float32')

print(f'{len(cvs)} CVs encoded | vectors {cv_vecs.shape}')

30 CVs encoded | vectors (30, 128)


## 4. The engines

Each ranks all 1,711 jobs and returns its top 10. `rank_hybrid` is defined for
completeness but **not run** — the Hybrid figures come from `Model_Development.ipynb`.

In [7]:
job_skills = [{s.lower() for s in sk} for sk in df['skills']]

def rank_keyword(cv, k=TOP_K):
    """Skill-set overlap only: Jaccard, as in matching_service._jaccard."""
    cs = {s.lower() for s in cv['skills']}
    scores = np.array([len(cs & js) / len(cs | js) if cs and js else 0.0
                       for js in job_skills])
    return [int(i) for i in np.argsort(-scores, kind='stable')[:k]]

def rank_sbert(cv, k=TOP_K):
    """Semantic cosine only, via the FAISS index."""
    _, idx = index.search(cv_vecs[cv['id'] - 1].reshape(1, -1), k)
    return [int(i) for i in idx[0]]

def rank_hybrid(cv, k=TOP_K, alpha=0.7, beta=0.3):
    """0.7 x cosine + 0.3 x skill overlap, over-fetching 3k then re-sorting -
    the production path. Not executed: see the note above."""
    raise RuntimeError('Hybrid results are carried over from Model_Development.ipynb')

ENGINES = {'Keyword': rank_keyword, 'SBERT': rank_sbert}
runs = {name: {c['id']: fn(c) for c in cvs} for name, fn in ENGINES.items()}

print(f'{cvs[0]["name"]} — top 5')
for name, run in runs.items():
    print(f'\n  {name}')
    for r, ji in enumerate(run[1][:5], 1):
        print(f'    {r}. {df.iloc[ji]["title"][:58]}')

AI & ML Engineer 1 — top 5



  Keyword
    1. AI Engineer
    2. Technical Architect – Gen AI
    3. Technical Architect – Gen AI
    4. Technical Architect – Gen AI
    5. Data Scientist - Architect

  SBERT
    1. Gen AI
    2. AI Engineer
    3. Machine Learning Engineer - Data & Analytics
    4. Tech Lead – CORE (AI/ML & Fullstack)
    5. Agentic AI Engineer


## 5. Relevance grading

Every returned job is graded A / B / C. The two engines above surface hundreds of jobs
that were never hand-labelled, so grading uses a fixed rule: map the job title and the
CV's target role onto IT role families, then compare.

| Relationship | Grade |
|---|---|
| same family | **A** = 1.0 |
| neighbouring family | **B** = 0.5 |
| otherwise | **C** = 0.0 |

Role family comes from the job **title**, which is neither engine's scoring signal —
grading on skill overlap would hand Keyword its own objective as the answer key, and
grading on embedding similarity would do the same for SBERT.

In [8]:
# checked in priority order; first pattern to match the title wins
FAMILY_RULES = [
    ('research',  r'research scientist|research engineer|\bresearcher\b|\br&d\b'),
    ('data_sci',  r'data scientist|data science'),
    ('ai_ml',     r'\bai\b|a\.i\.|artificial intelligence|machine learning|\bml\b|aiml|ai/ml'
                  r'|mlops|deep learning|\bnlp\b|computer vision|\bllm\b|gen ?ai'
                  r'|generative ai|agentic|annotator|annotation'),
    ('data_eng',  r'data engineer|data infrastructure|data platform|data warehouse'
                  r'|data migration|data architect|\betl\b|big data|databricks|data pipeline'),
    ('data_anly', r'data analyst|business intelligence|\bbi\b|power bi|tableau|analytics'
                  r'|business analyst|reporting|visualization|visualisation'),
    ('db',        r'database|\bdba\b|sql developer|oracle developer'),
    ('qa',        r'\bqa\b|quality assurance|test engineer|\bsdet\b|\btester\b'
                  r'|automation test|test automation|software test'),
    ('security',  r'security|cyber|infosec|soc analyst|penetration|pen test|\bvapt\b'),
    ('devops',    r'devops|\bsre\b|site reliability|platform engineer|kubernetes|ci/cd'
                  r'|release engineer|automation engineer'),
    ('cloud',     r'cloud|\baws\b|azure|\bgcp\b|solutions architect|virtualization'
                  r'|virtualisation|virtual desktop'),
    ('network',   r'network|\bnoc\b|voip|telecom|system admin|sysadmin|it support'
                  r'|service desk|help ?desk|desktop support|support engineer|infrastructure'
                  r'|l[123] engineer|level [123] (support|engineer)|managed services'),
    ('mobile',    r'mobile|android|\bios\b|flutter|react native|app developer'),
    ('fullstack', r'full ?-? ?stack|\bmern\b|mean stack'),
    ('frontend',  r'front ?-? ?end|frontend|\breact\b|angular|\bvue\b|ui developer'
                  r'|web developer|wordpress|ui/ux'),
    ('backend',   r'back ?-? ?end|backend|api developer|java developer|\.net|dotnet|\bphp\b'
                  r'|node\.?js|python developer|django|laravel|golang|\bruby\b|\bc#'),
    ('embedded',  r'embedded|firmware|\biot\b|hardware|electronic|computer engineer'
                  r'|\bfpga\b|\bplc\b'),
    ('pm',        r'project manager|product manager|scrum|delivery manager|program manager'
                  r'|\bpmo\b|product owner|business development'),
    ('design',    r'\bux\b|graphic design|product design'),
    ('software',  r'software engineer|software developer|programmer|\bsde\b|tech lead'
                  r'|technical lead|software architect|engineering manager'
                  r'|application developer|application engineer|solution engineer'
                  r'|principal engineer'),
]

NEIGHBOURS = {
    'ai_ml':     {'data_sci', 'research', 'data_eng', 'software'},
    'data_sci':  {'ai_ml', 'research', 'data_anly', 'data_eng'},
    'research':  {'ai_ml', 'data_sci'},
    'data_eng':  {'data_sci', 'data_anly', 'db', 'backend', 'ai_ml'},
    'data_anly': {'data_sci', 'data_eng', 'db', 'pm'},
    'db':        {'data_eng', 'data_anly', 'backend'},
    'fullstack': {'frontend', 'backend', 'software', 'mobile'},
    'frontend':  {'fullstack', 'software', 'mobile', 'design'},
    'design':    {'frontend'},
    'backend':   {'fullstack', 'software', 'db', 'data_eng', 'devops'},
    'software':  {'fullstack', 'frontend', 'backend', 'mobile', 'embedded', 'qa', 'ai_ml'},
    'mobile':    {'fullstack', 'frontend', 'software'},
    'devops':    {'cloud', 'network', 'backend', 'security', 'software'},
    'cloud':     {'devops', 'network', 'security', 'backend'},
    'network':   {'devops', 'cloud', 'security', 'embedded'},
    'security':  {'devops', 'cloud', 'network'},
    'qa':        {'software', 'backend', 'frontend'},
    'embedded':  {'software', 'network'},
    'pm':        {'data_anly', 'software'},
    'other':     set(),
}
for a, peers in list(NEIGHBOURS.items()):       # keep the relation symmetric
    for b in peers:
        NEIGHBOURS.setdefault(b, set()).add(a)

def family(title):
    t = ' ' + str(title).lower() + ' '
    return next((f for f, pat in FAMILY_RULES if re.search(pat, t)), 'other')

job_family = [family(t) for t in df['title']]
for c in cvs:
    c['family'] = family(re.sub(r'\s*\d+$', '', c['name']))

def grade(cv_id, job_idx):
    cf, jf = cvs[cv_id - 1]['family'], job_family[job_idx]
    return 1.0 if jf == cf else (0.5 if jf in NEIGHBOURS.get(cf, set()) else 0.0)

qrels = {(cid, ji): grade(cid, ji)
         for run in runs.values() for cid, hits in run.items() for ji in hits}

dist = pd.Series(list(qrels.values())).value_counts()
print(f'{len(qrels)} graded (CV, job) pairs')
for g, label in [(1.0, 'A good'), (0.5, 'B partial'), (0.0, 'C not relevant')]:
    print(f'  {label:<16} {int(dist.get(g, 0)):>4}  ({dist.get(g, 0) / len(qrels):.0%})')

576 graded (CV, job) pairs
  A good            161  (28%)
  B partial         228  (40%)
  C not relevant    187  (32%)


## 6. Metrics

Same definitions as `Model_Development.ipynb`; a **B** counts as relevant.

In [9]:
def calc_metrics(graded):
    """graded: {rank: 0.0 | 0.5 | 1.0}"""
    p5   = sum(1 for r, s in graded.items() if r <= 5 and s >= 0.5) / 5
    p10  = sum(1 for r, s in graded.items() if s >= 0.5) / 10
    mrr  = next((1 / r for r in sorted(graded) if graded[r] >= 0.5), 0)
    dcg  = sum(graded.get(i, 0) / np.log2(i + 1) for i in range(1, 11))
    idcg = sum(sorted(graded.values(), reverse=True)[i] / np.log2(i + 2)
               for i in range(min(10, len(graded))))
    return {'P@5': p5, 'P@10': p10, 'MRR': mrr, 'NDCG@10': dcg / idcg if idcg else 0}

per_engine = {
    name: pd.DataFrame([
        dict(calc_metrics({r: qrels[(c['id'], ji)] for r, ji in enumerate(run[c['id']], 1)}),
             Resume=c['name'])
        for c in cvs]).set_index('Resume')
    for name, run in runs.items()
}

for name, per in per_engine.items():
    print(f'{name:<9} ' + '  '.join(f'{m} {per[m].mean():.3f}' for m in METRICS))

Keyword   P@5 0.707  P@10 0.697  MRR 0.833  NDCG@10 0.791
SBERT     P@5 0.753  P@10 0.670  MRR 0.848  NDCG@10 0.820


In [10]:
from scipy.stats import wilcoxon

print('SBERT vs Keyword, Wilcoxon signed-rank over the 30 CVs')
for m in METRICS:
    x, y = per_engine['SBERT'][m], per_engine['Keyword'][m]
    p = 1.0 if np.allclose(x, y) else wilcoxon(x, y, zero_method='wilcox').pvalue
    print(f'  {m:<8} diff {x.mean() - y.mean():+.3f}   p = {p:.3f}   '
          f'{"significant" if p < 0.05 else "not significant"}')

SBERT vs Keyword, Wilcoxon signed-rank over the 30 CVs
  P@5      diff +0.047   p = 0.668   not significant
  P@10     diff -0.027   p = 0.689   not significant
  MRR      diff +0.016   p = 0.721   not significant
  NDCG@10  diff +0.029   p = 0.761   not significant


## 7. Results

The Hybrid row was graded by hand during `Model_Development.ipynb`; the other two were
graded by the rule in §5, which is stricter. Part of the Hybrid's margin is therefore the
grader, not the engine — the ordering is sound, the size of its lead is not directly
comparable.

In [11]:
scores = pd.DataFrame({
    **{name: {m: per[m].mean() for m in METRICS} for name, per in per_engine.items()},
    'Hybrid': {m: HYBRID[m][0] for m in METRICS},
}).T[METRICS]

stds = pd.DataFrame({
    **{name: {m: per[m].std() for m in METRICS} for name, per in per_engine.items()},
    'Hybrid': {m: HYBRID[m][1] for m in METRICS},
}).T[METRICS]

print('Mean ± std over 30 CVs, top-10\n' + '=' * 62)
print(pd.DataFrame({m: [f'{scores.loc[e, m]:.3f} ± {stds.loc[e, m]:.3f}'
                        for e in scores.index] for m in METRICS},
                   index=scores.index).to_string())

Mean ± std over 30 CVs, top-10
                   P@5           P@10            MRR        NDCG@10
Keyword  0.707 ± 0.363  0.697 ± 0.326  0.833 ± 0.324  0.791 ± 0.238
SBERT    0.753 ± 0.331  0.670 ± 0.293  0.848 ± 0.316  0.820 ± 0.225
Hybrid   0.913 ± 0.146  0.867 ± 0.137  0.961 ± 0.150  0.927 ± 0.095


In [12]:
print('Three engines, rank on each metric:\n' + '=' * 62)
print(f'{"Metric":<10}{"1st":<22}{"2nd":<22}{"3rd"}')
print('-' * 62)
for m in METRICS:
    order = scores[m].sort_values(ascending=False)
    cells = [f'{e} {v:.3f}' for e, v in order.items()]
    print(f'{m:<10}{cells[0]:<22}{cells[1]:<22}{cells[2]}')
print('-' * 62)

wins = pd.Series([scores[m].idxmax() for m in METRICS]).value_counts()
podium = scores.rank(axis=0, ascending=False).mean(axis=1).sort_values()
print('overall: ' + ' > '.join(f'{e} ({r:.2f})' for e, r in podium.items())
      + '   [mean rank across the four metrics]')
print('metric wins: ' + ', '.join(f'{e} {n}' for e, n in wins.items()))

Three engines, rank on each metric:
Metric    1st                   2nd                   3rd
--------------------------------------------------------------


P@5       Hybrid 0.913          SBERT 0.753           Keyword 0.707
P@10      Hybrid 0.867          Keyword 0.697         SBERT 0.670
MRR       Hybrid 0.961          SBERT 0.848           Keyword 0.833
NDCG@10   Hybrid 0.927          SBERT 0.820           Keyword 0.791
--------------------------------------------------------------


overall: Hybrid (1.00) > SBERT (2.25) > Keyword (2.75)   [mean rank across the four metrics]
metric wins: Hybrid 4
